# Fetch NLR (formerly NREL) EV Charger Stations (Canada)

## Background

The U.S. Department of Energy's **NLR** (formerly NREL — rebranded, and the API domain
moved from `developer.nrel.gov` to `developer.nlr.gov` in May 2026) maintains
the Alternative Fuel Stations database — the same data that powers the
[Alternative Fueling Station Locator](https://afdc.energy.gov/stations). It covers the
US **and** Canada.

Unlike StatCan's two-step WDS flow, this is a straightforward REST API:
one GET request with query parameters → JSON response with all matching stations.

**What you need before starting:**
1. A free API key — sign up at https://www.nlr.gov/signup/
2. `requests` (you already have this from the StatCan fetch)
3. `python-dotenv` — install with `pip install python-dotenv`

## What you'll learn
- Storing API keys safely with a `.env` file (so you never commit secrets to git)
- Passing query parameters to a REST API
- Inspecting a JSON response to understand its structure
- Saving raw JSON to disk

## Checkpoints
- [ ] API key loaded from `.env` (not hardcoded in the notebook)
- [ ] GET request returns status 200
- [ ] You can print how many stations were returned
- [ ] Raw JSON saved to `data/raw/nrel_chargers.json`

---
## Part 1 — Setup & API key

In [ ]:
import requests
import json
from pathlib import Path

In [ ]:
RAW_DIR = Path.cwd() / "data" / "raw"
print(RAW_DIR)

### TODO 1 — Load your API key from a `.env` file

API keys are like passwords — they should **never** appear in code that gets committed.
The standard pattern is:

1. Create a file called `.env` in the `fetch/` folder (same folder as this notebook)
2. Put your key in it: `NREL_API_KEY=your_actual_key_here`
3. Load it in Python with `python-dotenv`

**Steps:**
- Create the `.env` file manually in VS Code (File → New File → save as `.env`)
- Make sure `.env` is in `.gitignore` so it never gets committed (check hint below)
- Fill in the code cell below to load the key

In [ ]:
# TODO 1: Load your API key
#
# 1. Import dotenv:        from dotenv import load_dotenv
# 2. Also import:          import os
# 3. Call:                  load_dotenv()           — this reads your .env file
# 4. Get the key:          api_key = os.getenv("NREL_API_KEY")
# 5. Verify it loaded:     print(api_key[:5], "...")   — shows first 5 chars only
#
# If it prints None, your .env file isn't being found — check the path.



---
## Part 2 — Build and send the API request

### TODO 2 — Set up the query parameters

The endpoint is:
```
GET https://developer.nlr.gov/api/alt-fuel-stations/v1.json
```
(Note: docs and old tutorials may still show `developer.nrel.gov` — that domain
was retired May 2026 and will refuse connections. Use `nlr.gov`.)

You pass filters as **query parameters**. We want:

| Parameter | Value | Why |
|---|---|---|
| `api_key` | your key | Authentication |
| `fuel_type` | `ELEC` | Electric chargers only |
| `country` | `CA` | Canada only |
| `status` | `E` | Currently open ("E" = existing/available) |
| `limit` | `all` | Return every matching station (not just first 200) |

With `requests`, you can pass parameters as a dictionary — cleaner than
building the URL string by hand.

In [ ]:
# TODO 2: Build the params dictionary and make the request
#
# 1. Set the base URL (the endpoint above — note it ends in .json)
# 2. Create a dict called `params` with the 5 key-value pairs from the table
# 3. Use requests.get(url, params=params) to make the request
# 4. Print the status code — you want 200
#
# Example of passing params (don't copy this, adapt it):
#   resp = requests.get("https://example.com/api", params={"key": "val"})



---
## Part 3 — Explore the response

### TODO 3 — Parse the JSON and look around

The response body is JSON. When you call `.json()` on the response object,
you get a Python dictionary. The stations are inside a key called `"fuel_stations"` —
it's a list of dictionaries, one per station.

Explore the structure before saving — this is how you'll know what columns
you're working with in the SQL stages.

In [ ]:
# TODO 3: Explore the JSON response
#
# 1. Parse:             data = response.json()    (or whatever you named it)
# 2. Top-level keys:    print(data.keys())        — see what's in the response
# 3. Count stations:    print(len(data["fuel_stations"]))
# 4. Peek at one:       print(data["fuel_stations"][0])  — look at the first station
#
# Bonus: to see it formatted nicely, try:
#   print(json.dumps(data["fuel_stations"][0], indent=2))
#
# Take note of interesting fields — you'll see things like:
#   station_name, city, state, latitude, longitude, ev_level2_evse_num,
#   ev_dc_fast_num, ev_network, ev_connector_types, open_date, etc.



---
## Part 4 — Save the raw data

### TODO 4 — Save the full JSON response to disk

Same principle as your StatCan fetch: **raw is sacred**. Save the exact API
response so everything downstream can be rebuilt from it.

For JSON, use Python's built-in `json` module to write it. Unlike the StatCan
zip (which was binary), JSON is text — use `open()` with write mode.

In [ ]:
# TODO 4: Save the raw JSON
#
# 1. Build the output path:    out_path = RAW_DIR / "nrel_chargers.json"
# 2. Make sure the folder exists (you did this in StatCan — same pattern)
# 3. Write the JSON:
#       with open(out_path, "w", encoding="utf-8") as f:
#           json.dump(data, f)           — writes the full dict to the file
# 4. Verify:  print the file size to confirm it wrote
#       print(f"Saved: {out_path.stat().st_size / 1_000_000:.1f} MB")
#
# Expected size: roughly 15-25 MB (depends on how many stations exist right now)



---
## Part 5 — Quick sanity check

### TODO 5 — Verify you can read it back

A good habit: after saving, read the file back and confirm the data survived
the round trip. This catches encoding issues or truncated writes early.

In [ ]:
# TODO 5: Read it back and verify
#
# 1. Read:     with open(RAW_DIR / "nrel_chargers.json", encoding="utf-8") as f:
#                  check = json.load(f)
# 2. Count:    print(f"Stations loaded back: {len(check['fuel_stations'])}")
# 3. Compare:  does the count match what you got in TODO 3?
#
# If the counts match, you're done with the fetch!



---
## Hints (try the TODOs first!)

<details><summary><b>Hint — TODO 1:</b> .gitignore for .env</summary>

Open your project's `.gitignore` and add a line:
```
.env
```
This tells git to never track `.env` files anywhere in the repo.
If `.gitignore` already has it (many templates do), you're good.

</details>

<details><summary><b>Hint — TODO 1:</b> load_dotenv not finding the file</summary>

`load_dotenv()` looks for `.env` in the **current working directory**.
In Jupyter, that's usually wherever you launched the notebook from.
Print `Path.cwd()` and make sure your `.env` is in that folder.
Or give it the explicit path:
```python
load_dotenv(Path.cwd() / ".env")
```

</details>

<details><summary><b>Hint — TODO 2:</b> What the params dict looks like</summary>

```python
params = {
    "api_key":   api_key,
    "fuel_type": "ELEC",
    "country":   "CA",
    "status":    "E",
    "limit":     "all",
}
```

</details>

<details><summary><b>Hint — TODO 2:</b> Status code isn't 200</summary>

- **403**: your API key is wrong or missing. Check the `.env` value.
- **400**: a parameter name or value is wrong. Double-check spelling.
- **429**: you've hit the rate limit (1,000 requests/hour). Wait a bit.

</details>

<details><summary><b>Hint — TODO 4:</b> json.dump vs json.dumps</summary>

- `json.dump(data, file)` — writes to a **file object** (what you want here)
- `json.dumps(data)` — returns a **string** (useful for printing)

Don't mix them up! `dump` takes a second argument (the file).

</details>